<a href="https://colab.research.google.com/github/mn660875/Navttc-Challenges/blob/main/azure_worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Azure AI Services: Choosing and Using Them
**NAVTTC / NIAI AI/ML Trainer Cohort** (Week 7, Class 1). Plain Python. **Tasks 1 to 5 need no Azure account, no key, no internet.** Fill every `None` / `# YOUR CODE` area, then run top to bottom. Check cells print `OK` or `FIX`.

You practise the thinking around ready-made AI services: choosing one, estimating cost, reading a result, using confidence, and handling a key safely. The service responses here are **fake sample data** made up for practice, not real Azure output.

In [ ]:
import os, json
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))
print("ready")

## Task 1: A service chooser

Write `choose_family(job)` that returns one of: `"vision"`, `"language"`, `"speech"`, `"document"`, `"generative"`. Use simple keyword rules on the lowercase job text: words like `photo`, `image`, `camera` mean vision; `audio`, `call`, `voice` mean speech; `invoice`, `form`, `receipt` mean document; `email`, `review`, `ticket`, `text` mean language; anything else is generative. The order of your checks matters: think about `a photo of an invoice`.

In [1]:
def choose_family(job):
    j = job.lower()
    if any(w in j for w in ["invoice", "form", "receipt"]):
        return "document"
    if any(w in j for w in ["photo", "image", "camera"]):
        return "vision"
    if any(w in j for w in ["audio", "call", "voice"]):
        return "speech"
    if any(w in j for w in ["email", "review", "ticket", "text"]):
        return "language"
    return "generative"

In [ ]:
try:
    tests = {
     "read totals from scanned invoices": "document",
     "transcribe call centre audio": "speech",
     "tag damaged parcels in camera photos": "vision",
     "sort support tickets by mood": "language",
     "chat with our policy documents": "generative",
     "a photo of an invoice": "document",
    }
    for job, want in tests.items():
        check(job, choose_family(job) == want, f"expected {want}, got {choose_family(job)}")
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

## Task 2: Estimate a monthly bill

Services usually bill per use. Write `monthly_cost(calls, price_per_1000, free_calls=0)`: the first `free_calls` are free, the rest cost `price_per_1000` per 1,000 calls. The price below is a **placeholder**, not a real Azure price: read the live pricing page before using real numbers.

In [2]:
PRICE = 1.50        # placeholder: money per 1,000 calls
def monthly_cost(calls, price_per_1000, free_calls=0):
    billable_calls = max(0, calls - free_calls)
    return (billable_calls / 1000.0) * price_per_1000

In [ ]:
try:
    a = monthly_cost(10_000, PRICE)
    b = monthly_cost(10_000, PRICE, free_calls=5_000)
    c = monthly_cost(3_000, PRICE, free_calls=5_000)
    print(f"10,000 calls: {a:.2f}   with 5,000 free: {b:.2f}   3,000 calls with 5,000 free: {c:.2f}   (placeholder price)")
    check("no free calls", abs(a - 15.0) < 1e-9, "10 x 1.50")
    check("free calls reduce the bill", abs(b - 7.5) < 1e-9, "only 5,000 are billed")
    check("never negative", c == 0, "use max(0, ...)")
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

## Task 3: Read a (fake) document-service result

A document service returns JSON with fields and a confidence score for each. Below is a **made-up** response. Write `get_fields(result)` that returns a plain dict like `{"vendor": "Alpha Traders", "total": 5000.0}` using each field's `value`. Convert `total` to a float.

In [3]:
fake_result = {
  "document": "invoice-001",
  "fields": {
    "vendor": {"value": "Alpha Traders", "confidence": 0.98},
    "date":   {"value": "2026-10-03",   "confidence": 0.91},
    "total":  {"value": "5000",         "confidence": 0.62},
  },
}
def get_fields(result):
    fields = {}
    for k, v in result["fields"].items():
        val = v["value"]
        if k == "total":
            fields[k] = float(val)
        else:
            fields[k] = val
    return fields

In [ ]:
try:
    f = get_fields(fake_result)
    print(f)
    check("vendor", f["vendor"] == "Alpha Traders", "")
    check("total is a float", isinstance(f["total"], float) and f["total"] == 5000.0, "float('5000')")
    check("three fields", len(f) == 3, "")
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

## Task 4: Confidence and a human

The `total` above has confidence 0.62. Write `needs_review(result, threshold)` that returns the **list of field names** whose confidence is below `threshold`. Then decide a sensible threshold for money amounts and explain your choice in a markdown cell below.

In [4]:
def needs_review(result, threshold):
    flagged = []
    for k, v in result["fields"].items():
        if v["confidence"] < threshold:
            flagged.append(k)
    return flagged

In [ ]:
try:
    print(needs_review(fake_result, 0.80), needs_review(fake_result, 0.95), needs_review(fake_result, 0.50))
    check("0.80 flags only total", needs_review(fake_result, 0.80) == ["total"], "")
    check("0.95 flags date and total", sorted(needs_review(fake_result, 0.95)) == ["date", "total"], "")
    check("0.50 flags nothing", needs_review(fake_result, 0.50) == [], "")
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

**Your answer (2 sentences):** I would pick a confidence threshold of `0.95` for the `total` field, because financial transactions require high accuracy to avoid billing errors. While a higher threshold increases the human review cost as more flags are raised, it significantly mitigates the financial risks of processing incorrect invoice totals.

## Task 5: A safe request builder

Real calls need an endpoint and a key. Never type a key into a notebook. Write `get_settings()` that reads `AI_ENDPOINT` and `AI_KEY` from **environment variables** and raises `RuntimeError` with a clear message if either is missing. Write `mask(key)` that shows only the last 4 characters, for example `****abcd`. This task makes **no network call**.

In [5]:
def get_settings():
    endpoint = os.environ.get("AI_ENDPOINT")
    key = os.environ.get("AI_KEY")
    if not endpoint or not key:
        raise RuntimeError("AI_ENDPOINT or AI_KEY is missing from environment variables!")
    return endpoint, key
def mask(key):
    if len(key) <= 4:
        return key
    return "*" * (len(key) - 4) + key[-4:]

In [ ]:
try:
    os.environ.pop("AI_ENDPOINT", None); os.environ.pop("AI_KEY", None)
    try:
        get_settings(); raised = False
    except RuntimeError:
        raised = True
    check("missing settings raise an error", raised, "raise RuntimeError(...)")
    os.environ["AI_ENDPOINT"] = "https://example.invalid"; os.environ["AI_KEY"] = "demo-key-abcd"
    ep, k = get_settings()
    check("reads the values", ep == "https://example.invalid" and k == "demo-key-abcd", "")
    check("mask hides the key", mask(k) == "****abcd", "")
    os.environ.pop("AI_ENDPOINT"); os.environ.pop("AI_KEY")   # clean up the fake values
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

## Part B: portal look-around (needs Azure access)
Menu names change and these steps were **not run by the author**. Do a dry run first.
1. Open **ai.azure.com** (you can browse the catalog without signing in). Find the tools for **Vision, Language, Speech and Document** and note what each one's page says it does.
2. If you have access, try one demo with a **harmless sample** (a public image or sentence). Do not upload personal or customer data.
3. Write down: which service would you use for the group project, what input it takes, and one risk.

**Reflection (2 sentences each):**

* **Which of the five jobs from the slides was hardest to choose?**
Choosing between language and generative AI was the hardest since policy documents can be handled by both translation/summarization models or a chat-based LLM. Analyzing which service is more cost-effective and precise is key.

* **What data would you never send to a cloud service?**
I would never upload highly sensitive or private datasets, such as personal health records, financial credentials, passwords, or proprietary business intelligence. These must be kept secure locally or on dedicated, strictly audited private servers.